# Notebook 1 - API, collecte et nettoyage

## Question du projet
Comment la météo impacte-t-elle la régularité mensuelle des trains SNCF, selon les régions et selon le type de réseau ?

## Hypothèse de départ
Les mois marqués par davantage d'épisodes météo extrêmes sont aussi les mois où la régularité baisse et où les annulations montent.

## Ce que fait ce notebook
- télécharge ou recharge toutes les sources HTTP nécessaires ;
- harmonise les régions SNCF vers les régions actuelles ;
- construit un indicateur composite de stress météo ;
- prépare toutes les tables utilisées ensuite dans `02_analyse_et_figures.ipynb`.

## Ordre d'exécution
Exécuter d'abord ce notebook, puis `02_analyse_et_figures.ipynb`.


## APIs et endpoints utilisés

Ici, toutes les sources sont appelées avec `requests.get(...)`, puis mises en cache dans `data/raw`.

| Source | Type | Endpoint principal | Rôle dans le projet |
|---|---|---|---|
| `SNCF Open Data - regularite-mensuelle-ter` | HTTP CSV | téléchargement direct | base principale du projet |
| `SNCF Open Data - ponctualite-mensuelle-transilien` | HTTP CSV | téléchargement direct | ajout de l'Île-de-France |
| `SNCF Open Data - regularite-mensuelle-intercites` | HTTP CSV | téléchargement direct | comparaison avec un autre réseau |
| `SNCF Open Data - gares-de-voyageurs` | HTTP CSV | téléchargement direct | référentiel des gares |
| `SNCF Open Data - frequentation-gares` | HTTP CSV | téléchargement direct | variable de contexte réseau |
| `geo.api.gouv.fr / regions` | API JSON | `https://geo.api.gouv.fr/regions` | noms officiels, chef-lieu, coordonnées |
| `geo.api.gouv.fr / departements` | API JSON | `https://geo.api.gouv.fr/departements` | rattachement département à région |
| `Open-Meteo Archive API` | API JSON | `https://archive-api.open-meteo.com/v1/archive` | historique météo quotidien |

## Indicateur composite construit dans le projet

Nous construisons un **score mensuel de stress météo** :

`weather_stress_score = moyenne( max(0, z_pluie_forte), max(0, z_pluie_tres_forte), max(0, z_neige), max(0, z_vent_fort), max(0, z_chaleur), max(0, z_gel), max(0, z_orage) )`

L'idée est de comparer chaque mois à ce qui est habituel pour **le même mois** dans **la même région**, afin d'éviter de confondre saisonnalité normale et météo exceptionnellement défavorable.


In [34]:
from pathlib import Path
import time
from typing import List

import numpy as np
import pandas as pd
import requests

from config import (
    FIGURES_DIR,
    FREQUENTATION_RAW_PATH,
    GARES_RAW_PATH,
    GEO_DEPARTEMENTS_CACHE_PATH,
    GEO_REGIONS_CACHE_PATH,
    HTTP_SOURCE_DETAILS,
    INTERCITES_MONTHLY_PATH,
    INTERCITES_RAW_PATH,
    MERGED_PATH,
    METRO_REGION_NAMES,
    MODE_COMPARISON_PATH,
    NUMERIC_INTERCITES_COLUMNS,
    NUMERIC_TER_COLUMNS,
    PROCESSED_DIR,
    RAW_DIR,
    REGION_CONTEXT_PATH,
    SNCF_DATASET_URLS,
    STRESS_COMPONENTS,
    TER_CLEAN_PATH,
    TER_MONTHLY_PATH,
    TER_RAW_PATH,
    TRANSILIEN_MONTHLY_PATH,
    TRANSILIEN_RAW_PATH,
    WEATHER_CACHE_PATH,
    WEATHER_MONTHLY_PATH,
    WEATHER_REGION_DIR,
)
from utils import (
    add_monthly_gap,
    build_region_lookup,
    clean_uic,
    ensure_directories,
    extract_department_code_from_commune,
    extract_department_code_from_postal,
    normalize_columns,
    normalize_text,
    shorten_comment,
    slugify,
    to_numeric,
    weighted_average,
    zscore_by_region_month,
)


In [35]:
def request_json(url: str, params: dict | None = None, attempts: int = 6, sleep_base: float = 4.0):
    last_error = None
    for attempt in range(attempts):
        try:
            response = requests.get(url, params=params, timeout=180)
            if response.status_code == 200:
                return response.json()
            if response.status_code in {429, 500, 502, 503, 504}:
                time.sleep(sleep_base * (attempt + 1))
                continue
            response.raise_for_status()
        except requests.RequestException as exc:
            last_error = exc
            time.sleep(sleep_base * (attempt + 1))
    raise RuntimeError(f"Échec de la requête vers {url}") from last_error


def fetch_current_regions_reference(force_refresh: bool = False) -> pd.DataFrame:
    if GEO_REGIONS_CACHE_PATH.exists() and not force_refresh:
        cached = pd.read_csv(GEO_REGIONS_CACHE_PATH)
        official_lookup = {normalize_text(name): name for name in METRO_REGION_NAMES}
        cached["region_current"] = cached["region_current"].map(
            lambda value: official_lookup.get(normalize_text(value), value)
        )
        return cached[cached["region_current"].isin(METRO_REGION_NAMES)].copy()

    regions = request_json("https://geo.api.gouv.fr/regions")
    rows: list[dict] = []
    for region in regions:
        detail = request_json(
            f"https://geo.api.gouv.fr/regions/{region['code']}",
            params={"fields": "nom,code,chefLieu"},
        )
        chef_lieu_code = detail.get("chefLieu")
        if not chef_lieu_code:
            continue
        commune = request_json(
            f"https://geo.api.gouv.fr/communes/{chef_lieu_code}",
            params={"fields": "nom,centre"},
        )
        coords = (commune.get("centre") or {}).get("coordinates") or [None, None]
        lon, lat = coords[0], coords[1]
        rows.append(
            {
                "region_code": detail["code"],
                "region_current": detail["nom"],
                "representative_city": commune.get("nom"),
                "latitude": lat,
                "longitude": lon,
            }
        )
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    df = df[df["region_current"].isin(METRO_REGION_NAMES)].copy().sort_values("region_current")
    df.to_csv(GEO_REGIONS_CACHE_PATH, index=False)
    return df


def fetch_departements_reference(force_refresh: bool = False) -> pd.DataFrame:
    if GEO_DEPARTEMENTS_CACHE_PATH.exists() and not force_refresh:
        cached = pd.read_csv(GEO_DEPARTEMENTS_CACHE_PATH)
        official_lookup = {normalize_text(name): name for name in METRO_REGION_NAMES}
        cached["region_current"] = cached["region_current"].map(
            lambda value: official_lookup.get(normalize_text(value), value)
        )
        return cached

    departments = request_json(
        "https://geo.api.gouv.fr/departements",
        params={"fields": "nom,code,region"},
    )
    rows: list[dict] = []
    for department in departments:
        region = department.get("region") or {}
        rows.append(
            {
                "departement_code": department.get("code"),
                "departement_name": department.get("nom"),
                "region_code": region.get("code"),
                "region_current": region.get("nom"),
            }
        )
    df = pd.DataFrame(rows)
    if "20" not in set(df["departement_code"]):
        df = pd.concat(
            [
                df,
                pd.DataFrame(
                    [
                        {
                            "departement_code": "20",
                            "departement_name": "Corse",
                            "region_code": "94",
                            "region_current": "Corse",
                        }
                    ]
                ),
            ],
            ignore_index=True,
        )
    df.to_csv(GEO_DEPARTEMENTS_CACHE_PATH, index=False)
    return df


def download_http_file(path: Path, url: str, force_refresh: bool = False) -> dict:
    if path.exists() and not force_refresh:
        return {
            "source_file": path.name,
            "cache_path": str(path),
            "mode": "cache",
            "url": url,
            "size_kb": round(path.stat().st_size / 1024, 1),
        }

    response = requests.get(url, timeout=180)
    response.raise_for_status()
    path.write_bytes(response.content)
    return {
        "source_file": path.name,
        "cache_path": str(path),
        "mode": "telechargement",
        "url": url,
        "size_kb": round(path.stat().st_size / 1024, 1),
    }


def seed_region_weather_caches_from_combined() -> None:
    if not WEATHER_CACHE_PATH.exists():
        return
    combined = pd.read_csv(WEATHER_CACHE_PATH)
    if "date" not in combined.columns or "region_current" not in combined.columns:
        return
    combined["date"] = pd.to_datetime(combined["date"], errors="coerce")
    for region_name, frame in combined.dropna(subset=["date", "region_current"]).groupby("region_current"):
        cache_path = WEATHER_REGION_DIR / f"{slugify(region_name)}.csv"
        if not cache_path.exists():
            frame.sort_values("date").to_csv(cache_path, index=False)


## 1. Collecte HTTP et cache local

La logique ci-dessous rend le notebook autonome :
- si les fichiers bruts existent déjà, on les recharge ;
- sinon ils sont téléchargés automatiquement ;
- les réponses API sont ensuite conservées dans `data/raw`.


In [36]:
ensure_directories()

FORCE_HTTP_REFRESH = False
FORCE_GEO_REFRESH = False
FORCE_WEATHER_REFRESH = False

download_logs: list[dict] = []
for path, url in SNCF_DATASET_URLS.items():
    row = download_http_file(path, url, force_refresh=FORCE_HTTP_REFRESH)
    row["source"] = next(item["source"] for item in HTTP_SOURCE_DETAILS if item["cache_path"] == path)
    row["role"] = next(item["role"] for item in HTTP_SOURCE_DETAILS if item["cache_path"] == path)
    download_logs.append(row)

source_catalog = pd.DataFrame(HTTP_SOURCE_DETAILS)[["source", "role", "url", "cache_path"]]
download_log = pd.DataFrame(download_logs)[["source", "role", "mode", "size_kb", "cache_path"]]

print("Sources HTTP suivies :", len(source_catalog))
source_catalog


Sources HTTP suivies : 8


,source,role,url,cache_path
0,SNCF TER,Régularité mensuelle TER,https://ressources.data.sncf.com/explore/datas...,C:\Users\antoc\analyse-impact-meteo-regularite...
1,SNCF Transilien,Ponctualité mensuelle Transilien,https://ressources.data.sncf.com/explore/datas...,C:\Users\antoc\analyse-impact-meteo-regularite...
2,SNCF Intercités,Régularité mensuelle Intercités,https://ressources.data.sncf.com/explore/datas...,C:\Users\antoc\analyse-impact-meteo-regularite...
3,SNCF Gares,Référentiel gares de voyageurs,https://ressources.data.sncf.com/explore/datas...,C:\Users\antoc\analyse-impact-meteo-regularite...
4,SNCF Fréquentation,Fréquentation des gares,https://ressources.data.sncf.com/explore/datas...,C:\Users\antoc\analyse-impact-meteo-regularite...
5,geo.api.gouv.fr / regions,Régions officielles,https://geo.api.gouv.fr/regions,C:\Users\antoc\analyse-impact-meteo-regularite...
6,geo.api.gouv.fr / departements,Départements officiels,https://geo.api.gouv.fr/departements?fields=no...,C:\Users\antoc\analyse-impact-meteo-regularite...
7,Open-Meteo Archive,Historique météo quotidien,https://archive-api.open-meteo.com/v1/archive,C:\Users\antoc\analyse-impact-meteo-regularite...


In [37]:
download_log


,source,role,mode,size_kb,cache_path
0,SNCF TER,Régularité mensuelle TER,cache,440.2,C:\Users\antoc\analyse-impact-meteo-regularite...
1,SNCF Transilien,Ponctualité mensuelle Transilien,cache,86.8,C:\Users\antoc\analyse-impact-meteo-regularite...
2,SNCF Intercités,Régularité mensuelle Intercités,cache,401.6,C:\Users\antoc\analyse-impact-meteo-regularite...
3,SNCF Gares,Référentiel gares de voyageurs,cache,254.9,C:\Users\antoc\analyse-impact-meteo-regularite...
4,SNCF Fréquentation,Fréquentation des gares,cache,533.5,C:\Users\antoc\analyse-impact-meteo-regularite...


## 2. Géographie officielle et harmonisation des régions SNCF

On combine ici :
- les régions actuelles issues de `geo.api.gouv.fr` ;
- un petit dictionnaire d'alias SNCF pour les anciens noms de régions ou les zones TER spécifiques.


In [38]:
region_reference = fetch_current_regions_reference(force_refresh=FORCE_GEO_REFRESH)
departements_reference = fetch_departements_reference(force_refresh=FORCE_GEO_REFRESH)
region_lookup = build_region_lookup(region_reference)

print("Régions officielles chargées :", region_reference.shape[0])
print("Départements chargés :", departements_reference.shape[0])
region_reference


Régions officielles chargées : 13
Départements chargés : 102


,region_code,region_current,representative_city,latitude,longitude
0,84,Auvergne-Rhône-Alpes,Lyon,45.7580,4.8351
1,27,Bourgogne-Franche-Comté,Dijon,47.3319,5.0322
2,53,Bretagne,Rennes,48.1159,-1.6884
3,24,Centre-Val de Loire,Orléans,47.8734,1.9122
4,94,Corse,Ajaccio,41.9228,8.7058
5,44,Grand Est,Strasbourg,48.5691,7.7621
6,32,Hauts-de-France,Lille,50.6311,3.0468
7,11,Île-de-France,Paris,48.8589,2.3470
8,28,Normandie,Rouen,49.4412,1.0912
9,75,Nouvelle-Aquitaine,Bordeaux,44.8624,-0.5848


## 3. TER : nettoyage, harmonisation et table mensuelle principale


In [39]:
def load_ter_monthly(region_lookup: dict) -> pd.DataFrame:
    raw = pd.read_csv(TER_RAW_PATH, sep=";", encoding="utf-8-sig")
    raw.columns = normalize_columns(raw.columns)
    raw["date"] = pd.to_datetime(raw["date"], format="%Y-%m", errors="coerce").dt.to_period("M").dt.to_timestamp()
    for column in NUMERIC_TER_COLUMNS:
        raw[column] = to_numeric(raw[column])
    raw["region_current"] = raw["region"].map(lambda value: region_lookup.get(normalize_text(value)))
    raw = raw.dropna(subset=["date", "region_current"]).copy()

    clean_columns = [
        "date",
        "region",
        "nombre_de_trains_programmes",
        "nombre_de_trains_ayant_circule",
        "nombre_de_trains_annules",
        "nombre_de_trains_en_retard_a_l_arrivee",
        "taux_de_regularite",
        "nombre_de_trains_a_l_heure_pour_un_train_en_retard_a_l_arrivee",
        "commentaires",
    ]
    raw[clean_columns].sort_values(["date", "region"]).to_csv(TER_CLEAN_PATH, index=False)

    rows: list[dict] = []
    for (date, region_current), group in raw.groupby(["date", "region_current"], sort=True):
        row = {
            "date": date,
            "region_current": region_current,
            "source_regions": ", ".join(sorted(group["region"].dropna().unique())),
        }
        for column in NUMERIC_TER_COLUMNS[:-1]:
            row[column] = np.nan if group[column].isna().any() else float(group[column].sum())
        comments = [str(value) for value in group["commentaires"].dropna().unique() if str(value).strip()]
        row["commentaires"] = " | ".join(comments)
        rows.append(row)

    ter = pd.DataFrame(rows).sort_values(["date", "region_current"]).reset_index(drop=True)
    ter["regularity_pct"] = 100 * (
        ter["nombre_de_trains_ayant_circule"] - ter["nombre_de_trains_en_retard_a_l_arrivee"]
    ) / ter["nombre_de_trains_ayant_circule"]
    ter["cancellation_pct"] = 100 * ter["nombre_de_trains_annules"] / ter["nombre_de_trains_programmes"]
    ter["delay_pct"] = 100 * ter["nombre_de_trains_en_retard_a_l_arrivee"] / ter["nombre_de_trains_ayant_circule"]
    ter["month"] = ter["date"].dt.month
    ter["year"] = ter["date"].dt.year
    ter["regularity_gap"] = add_monthly_gap(ter, "regularity_pct", ["region_current"])
    ter["cancellation_gap"] = add_monthly_gap(ter, "cancellation_pct", ["region_current"])
    ter.to_csv(TER_MONTHLY_PATH, index=False)
    return ter


ter_monthly = load_ter_monthly(region_lookup)
analysis_start_date = ter_monthly["date"].min().strftime("%Y-%m-%d")
analysis_end_date = (ter_monthly["date"].max() + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")

ter_regions = sorted(ter_monthly["region_current"].dropna().unique())
weather_regions = sorted(set(ter_regions) | {"Île-de-France"})
region_reference_target = (
    region_reference[region_reference["region_current"].isin(weather_regions)]
    .copy()
    .sort_values("region_current")
)

print("Période TER :", analysis_start_date, "->", analysis_end_date)
print("Régions TER :", len(ter_regions))
print("Régions météo nécessaires :", len(weather_regions))
ter_monthly.head()


Période TER : 2013-01-01 -> 2026-02-28
Régions TER : 11
Régions météo nécessaires : 12


,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,delay_pct,month,year,regularity_gap,cancellation_gap
0,2013-01-01,Auvergne-Rhône-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,10.909041,1,2013,-1.286416,-0.075489
1,2013-01-01,Bourgogne-Franche-Comté,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,8.368855,1,2013,-1.302409,-0.399500
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,6.418723,1,2013,-1.342716,0.126337
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,8.382368,1,2013,0.544582,0.042994
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,NaN,1,2013,NaN,NaN


## 4. Open-Meteo : historique quotidien et score mensuel de stress météo

Pour rendre la chaîne robuste, on met en cache **une réponse par région**.
Cela évite de refaire toute la collecte à chaque exécution et permet aussi de compléter les régions manquantes si besoin.


In [40]:
def load_region_weather(row: pd.Series, start_date: str, end_date: str, force_refresh: bool = False) -> pd.DataFrame:
    cache_path = WEATHER_REGION_DIR / f"{slugify(row['region_current'])}.csv"

    if cache_path.exists() and not force_refresh:
        cached = pd.read_csv(cache_path)
        cached["date"] = pd.to_datetime(cached["date"], errors="coerce")
        if (
            not cached.empty
            and cached["date"].min() <= pd.Timestamp(start_date)
            and cached["date"].max() >= pd.Timestamp(end_date)
        ):
            return cached

    payload = request_json(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "start_date": start_date,
            "end_date": end_date,
            "daily": ",".join(
                [
                    "temperature_2m_mean",
                    "temperature_2m_max",
                    "temperature_2m_min",
                    "precipitation_sum",
                    "snowfall_sum",
                    "wind_speed_10m_max",
                    "wind_gusts_10m_max",
                    "weather_code",
                ]
            ),
            "timezone": "Europe/Paris",
        },
    )

    daily = pd.DataFrame(payload["daily"]).rename(columns={"time": "date"})
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily["region_current"] = row["region_current"]
    daily["representative_city"] = row["representative_city"]
    daily = daily.sort_values("date").reset_index(drop=True)
    daily.to_csv(cache_path, index=False)
    time.sleep(0.2)
    return daily


def load_weather_daily(region_reference_target: pd.DataFrame, start_date: str, end_date: str, force_refresh: bool = False) -> pd.DataFrame:
    seed_region_weather_caches_from_combined()
    frames: list[pd.DataFrame] = []
    for _, row in region_reference_target.reset_index(drop=True).iterrows():
        frame = load_region_weather(row, start_date=start_date, end_date=end_date, force_refresh=force_refresh)
        frames.append(frame)

    daily = pd.concat(frames, ignore_index=True)
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily = daily.dropna(subset=["date", "region_current"]).copy()
    daily = daily[(daily["date"] >= pd.Timestamp(start_date)) & (daily["date"] <= pd.Timestamp(end_date))].copy()
    daily.to_csv(WEATHER_CACHE_PATH, index=False)

    expected_regions = set(region_reference_target["region_current"])
    actual_regions = set(daily["region_current"])
    missing_regions = sorted(expected_regions - actual_regions)
    if missing_regions:
        raise RuntimeError(f"Couverture météo incomplète : {missing_regions}")
    return daily


def build_weather_monthly(daily: pd.DataFrame) -> pd.DataFrame:
    weather = daily.copy()
    weather["month_start"] = weather["date"].dt.to_period("M").dt.to_timestamp()
    weather["month"] = weather["date"].dt.month

    weather["heavy_rain_day"] = (weather["precipitation_sum"] >= 10).astype(int)
    weather["very_heavy_rain_day"] = (weather["precipitation_sum"] >= 20).astype(int)
    weather["snow_day"] = (weather["snowfall_sum"] > 0.1).astype(int)
    weather["strong_wind_day"] = (weather["wind_gusts_10m_max"] >= 70).astype(int)
    weather["heat_day"] = (weather["temperature_2m_max"] >= 30).astype(int)
    weather["frost_day"] = (weather["temperature_2m_min"] <= 0).astype(int)
    weather["storm_day"] = weather["weather_code"].isin([95, 96, 99]).astype(int)

    monthly = (
        weather.groupby(["month_start", "region_current"], as_index=False)
        .agg(
            representative_city=("representative_city", "first"),
            temp_mean=("temperature_2m_mean", "mean"),
            temp_max_peak=("temperature_2m_max", "max"),
            temp_min_low=("temperature_2m_min", "min"),
            precipitation_total=("precipitation_sum", "sum"),
            snowfall_total=("snowfall_sum", "sum"),
            heavy_rain_days=("heavy_rain_day", "sum"),
            very_heavy_rain_days=("very_heavy_rain_day", "sum"),
            snow_days=("snow_day", "sum"),
            strong_wind_days=("strong_wind_day", "sum"),
            heat_days=("heat_day", "sum"),
            frost_days=("frost_day", "sum"),
            storm_days=("storm_day", "sum"),
        )
        .rename(columns={"month_start": "date"})
        .sort_values(["date", "region_current"])
        .reset_index(drop=True)
    )

    monthly["month"] = monthly["date"].dt.month
    monthly["year"] = monthly["date"].dt.year

    for metric in STRESS_COMPONENTS:
        monthly[f"{metric}_z"] = monthly.groupby(["region_current", "month"])[metric].transform(zscore_by_region_month)
        monthly[f"{metric}_stress"] = monthly[f"{metric}_z"].clip(lower=0)

    stress_columns = [f"{metric}_stress" for metric in STRESS_COMPONENTS]
    monthly["weather_stress_score"] = monthly[stress_columns].mean(axis=1)
    monthly["dominant_weather_key"] = monthly[stress_columns].idxmax(axis=1).str.replace("_stress", "", regex=False)
    monthly["dominant_weather_label"] = monthly["dominant_weather_key"].map(STRESS_COMPONENTS)
    monthly.to_csv(WEATHER_MONTHLY_PATH, index=False)
    return monthly


weather_daily = load_weather_daily(
    region_reference_target,
    start_date=analysis_start_date,
    end_date=analysis_end_date,
    force_refresh=FORCE_WEATHER_REFRESH,
)
weather_monthly = build_weather_monthly(weather_daily)

weather_coverage = (
    weather_monthly.groupby("region_current")["date"]
    .agg(["min", "max", "count"])
    .reset_index()
    .sort_values("region_current")
)
weather_coverage


,region_current,min,max,count
0,Auvergne-Rhône-Alpes,2013-01-01,2026-02-01,158
1,Bourgogne-Franche-Comté,2013-01-01,2026-02-01,158
2,Bretagne,2013-01-01,2026-02-01,158
3,Centre-Val de Loire,2013-01-01,2026-02-01,158
4,Grand Est,2013-01-01,2026-02-01,158
5,Hauts-de-France,2013-01-01,2026-02-01,158
6,Normandie,2013-01-01,2026-02-01,158
7,Nouvelle-Aquitaine,2013-01-01,2026-02-01,158
8,Occitanie,2013-01-01,2026-02-01,158
9,Pays de la Loire,2013-01-01,2026-02-01,158


## 5. Fusion TER + météo

La table fusionnée reste la base principale de l'analyse.
Elle servira à la fois aux graphiques TER et à la détection des épisodes météo marquants.


In [41]:
def merge_datasets(ter: pd.DataFrame, weather_monthly: pd.DataFrame) -> pd.DataFrame:
    merged = ter.merge(weather_monthly, on=["date", "region_current", "month", "year"], how="left")
    q20 = merged["weather_stress_score"].quantile(0.20)
    q80 = merged["weather_stress_score"].quantile(0.80)
    merged["stress_bucket"] = pd.cut(
        merged["weather_stress_score"],
        bins=[-1e9, q20, q80, 1e9],
        labels=["Mois calmes", "Mois intermédiaires", "Mois chocs météo"],
        include_lowest=True,
    )
    merged["comment_clean"] = merged["commentaires"].map(shorten_comment)
    merged.to_csv(MERGED_PATH, index=False)
    return merged


merged = merge_datasets(ter_monthly, weather_monthly)
print("Base TER + météo :", merged.shape)
merged.head()


Base TER + météo : (1738, 47)


,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean
0,2013-01-01,Auvergne-Rhône-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,...,0.0,0.897127,0.897127,0.0,0.0,0.187785,frost_days,Gel,Mois intermédiaires,Conditions météos défavorables.
1,2013-01-01,Bourgogne-Franche-Comté,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,...,0.0,0.498754,0.498754,0.0,0.0,0.279451,snow_days,Neige,Mois intermédiaires,Un mois de janvier qui surpasse les six exerci...
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,...,0.0,0.404067,0.404067,0.0,0.0,0.339054,snow_days,Neige,Mois intermédiaires,Fortes chutes de neige ayant entrainé des pert...
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,...,0.0,0.550090,0.550090,0.0,0.0,0.412076,snow_days,Neige,Mois intermédiaires,"Trois incidents caténaires lourds, dont deux p..."
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,...,0.0,0.438215,0.438215,0.0,0.0,0.155910,snow_days,Neige,Mois intermédiaires,Conditions météorologiques dégradées du 15 au ...


## 6. Réseaux comparés : Transilien et Intercités

Comme les indicateurs ne sont pas exactement les mêmes selon les réseaux, on crée ensuite une table de comparaison basée sur :
- le niveau mensuel de performance ;
- l'écart au niveau habituel du même mois (`performance_gap`) ;
- un score météo adapté au périmètre du réseau.


In [42]:
def load_transilien_monthly(weather_monthly: pd.DataFrame) -> pd.DataFrame:
    raw = pd.read_csv(TRANSILIEN_RAW_PATH, sep=";", encoding="utf-8-sig")
    raw.columns = normalize_columns(raw.columns)
    raw["date"] = pd.to_datetime(raw["date"], format="%Y-%m", errors="coerce").dt.to_period("M").dt.to_timestamp()
    raw["taux_de_ponctualite"] = to_numeric(raw["taux_de_ponctualite"])
    raw = raw.dropna(subset=["date", "taux_de_ponctualite"]).copy()

    monthly = (
        raw.groupby("date", as_index=False)
        .agg(
            performance_pct=("taux_de_ponctualite", "mean"),
            line_count=("ligne", pd.Series.nunique),
            service_count=("service", pd.Series.nunique),
            observation_count=("ligne", "count"),
        )
        .sort_values("date")
        .reset_index(drop=True)
    )
    monthly["month"] = monthly["date"].dt.month
    monthly["year"] = monthly["date"].dt.year
    monthly["performance_gap"] = add_monthly_gap(monthly, "performance_pct", [])
    monthly["mode"] = "Transilien"
    monthly["metric_label"] = "Ponctualité voyageurs"
    monthly["cancellation_pct"] = np.nan
    monthly["weather_scope"] = "Île-de-France"

    idf_weather = weather_monthly.loc[
        weather_monthly["region_current"] == "Île-de-France",
        ["date", "weather_stress_score"],
    ].copy()
    monthly = monthly.merge(idf_weather, on="date", how="left")
    monthly.to_csv(TRANSILIEN_MONTHLY_PATH, index=False)
    return monthly


def load_intercites_monthly(weather_monthly: pd.DataFrame) -> pd.DataFrame:
    raw = pd.read_csv(INTERCITES_RAW_PATH, sep=";", encoding="utf-8-sig")
    raw.columns = normalize_columns(raw.columns)
    raw["date"] = pd.to_datetime(raw["date"], format="%Y-%m", errors="coerce").dt.to_period("M").dt.to_timestamp()
    for column in NUMERIC_INTERCITES_COLUMNS:
        raw[column] = to_numeric(raw[column])
    raw = raw.dropna(subset=["date"]).copy()

    monthly = (
        raw.groupby("date", as_index=False)
        .agg(
            nombre_de_trains_programmes=("nombre_de_trains_programmes", "sum"),
            nombre_de_trains_ayant_circule=("nombre_de_trains_ayant_circule", "sum"),
            nombre_de_trains_annules=("nombre_de_trains_annules", "sum"),
            nombre_de_trains_en_retard_a_l_arrivee=("nombre_de_trains_en_retard_a_l_arrivee", "sum"),
            relation_count=("depart", "count"),
            observation_count=("depart", "count"),
        )
        .sort_values("date")
        .reset_index(drop=True)
    )
    monthly["performance_pct"] = 100 * (
        monthly["nombre_de_trains_ayant_circule"] - monthly["nombre_de_trains_en_retard_a_l_arrivee"]
    ) / monthly["nombre_de_trains_ayant_circule"]
    monthly["cancellation_pct"] = 100 * monthly["nombre_de_trains_annules"] / monthly["nombre_de_trains_programmes"]
    monthly["month"] = monthly["date"].dt.month
    monthly["year"] = monthly["date"].dt.year
    monthly["performance_gap"] = add_monthly_gap(monthly, "performance_pct", [])
    monthly["mode"] = "Intercités"
    monthly["metric_label"] = "Régularité composite"
    monthly["weather_scope"] = "France métropolitaine"

    national_weather = (
        weather_monthly.groupby("date", as_index=False)
        .agg(weather_stress_score=("weather_stress_score", "mean"))
    )
    monthly = monthly.merge(national_weather, on="date", how="left")
    monthly.to_csv(INTERCITES_MONTHLY_PATH, index=False)
    return monthly


transilien_monthly = load_transilien_monthly(weather_monthly)
intercites_monthly = load_intercites_monthly(weather_monthly)

print("Transilien :", transilien_monthly.shape)
print("Intercités :", intercites_monthly.shape)
transilien_monthly.head()


Transilien : (153, 13)
Intercités : (139, 16)


,date,performance_pct,line_count,service_count,observation_count,month,year,performance_gap,mode,metric_label,cancellation_pct,weather_scope,weather_stress_score
0,2013-01-01,86.315385,13,2,13,1,2013,-2.638812,Transilien,Ponctualité voyageurs,NaN,Île-de-France,0.380085
1,2013-02-01,87.153846,13,2,13,2,2013,-1.900623,Transilien,Ponctualité voyageurs,NaN,Île-de-France,0.621583
2,2013-03-01,86.100000,13,2,13,3,2013,-4.360348,Transilien,Ponctualité voyageurs,NaN,Île-de-France,0.660999
3,2013-04-01,88.138462,13,2,13,4,2013,-2.699636,Transilien,Ponctualité voyageurs,NaN,Île-de-France,0.285121
4,2013-05-01,90.723077,13,2,13,5,2013,0.027301,Transilien,Ponctualité voyageurs,NaN,Île-de-France,0.000000


## 7. Variables de contexte : gares et fréquentation


In [43]:
def build_region_context(departements_reference: pd.DataFrame, target_regions: list[str]) -> pd.DataFrame:
    gares = pd.read_csv(GARES_RAW_PATH, sep=";", encoding="utf-8-sig")
    gares.columns = normalize_columns(gares.columns)
    gares["code_uic_clean"] = gares["code_uic"].map(clean_uic)
    gares["departement_code"] = gares["code_commune"].map(extract_department_code_from_commune)
    gares = gares.merge(
        departements_reference[["departement_code", "region_current"]].drop_duplicates(),
        on="departement_code",
        how="left",
    )
    gares = gares.dropna(subset=["code_uic_clean", "region_current"]).copy()

    frequentation = pd.read_csv(FREQUENTATION_RAW_PATH, sep=";", encoding="utf-8-sig")
    frequentation.columns = normalize_columns(frequentation.columns)
    frequentation["code_uic_clean"] = frequentation["code_uic"].map(clean_uic)
    frequentation["departement_code"] = frequentation["code_postal"].map(extract_department_code_from_postal)
    for column in ["total_voyageurs_2024", "total_voyageurs_2019"]:
        frequentation[column] = to_numeric(frequentation[column]).fillna(0)

    gares_ref = gares[["code_uic_clean", "region_current"]].drop_duplicates()
    frequentation = frequentation.merge(gares_ref, on="code_uic_clean", how="left")
    frequentation = frequentation.merge(
        departements_reference[["departement_code", "region_current"]]
        .drop_duplicates()
        .rename(columns={"region_current": "region_from_postal"}),
        on="departement_code",
        how="left",
    )
    frequentation["region_current"] = frequentation["region_current"].fillna(frequentation["region_from_postal"])
    frequentation = frequentation.dropna(subset=["region_current"]).copy()

    context = (
        gares.groupby("region_current", as_index=False)
        .agg(gare_count=("code_uic_clean", pd.Series.nunique))
        .merge(
            frequentation.groupby("region_current", as_index=False).agg(
                gares_avec_frequentation=("code_uic_clean", pd.Series.nunique),
                frequentation_voyageurs_2024=("total_voyageurs_2024", "sum"),
                frequentation_voyageurs_2019=("total_voyageurs_2019", "sum"),
            ),
            on="region_current",
            how="left",
        )
    )
    context = context[context["region_current"].isin(target_regions)].copy()
    context["gares_avec_frequentation"] = context["gares_avec_frequentation"].fillna(0).astype(int)
    context["frequentation_voyageurs_2024"] = context["frequentation_voyageurs_2024"].fillna(0)
    context["frequentation_voyageurs_2019"] = context["frequentation_voyageurs_2019"].fillna(0)
    context["voyageurs_par_gare_2024"] = context["frequentation_voyageurs_2024"] / context["gare_count"]
    context["frequentation_2024_vs_2019_pct"] = np.where(
        context["frequentation_voyageurs_2019"] > 0,
        100 * (context["frequentation_voyageurs_2024"] - context["frequentation_voyageurs_2019"]) / context["frequentation_voyageurs_2019"],
        np.nan,
    )
    context = context.sort_values("region_current").reset_index(drop=True)
    context.to_csv(REGION_CONTEXT_PATH, index=False)
    return context


region_context = build_region_context(
    departements_reference=departements_reference,
    target_regions=sorted(set(ter_regions) | {"Île-de-France"}),
)
region_context.head()


,region_current,gare_count,gares_avec_frequentation,frequentation_voyageurs_2024,frequentation_voyageurs_2019,voyageurs_par_gare_2024,frequentation_2024_vs_2019_pct
0,Auvergne-Rhône-Alpes,274,289,152459767,121844858,556422.507299,25.126140
1,Bourgogne-Franche-Comté,204,213,40183605,31663519,196978.455882,26.908209
2,Bretagne,127,134,46027471,31738072,362421.031496,45.022896
3,Centre-Val de Loire,162,163,48309624,36822952,298207.555556,31.194327
4,Grand Est,325,402,126758698,105059190,390026.763077,20.654555


## 8. Table finale de comparaison des modes

Cette table réunit TER, Transilien et Intercités dans un format unique pour les figures comparatives du notebook 2.


In [44]:
def build_ter_mode_monthly(merged: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict] = []
    for date, group in merged.groupby("date", sort=True):
        programmed = group["nombre_de_trains_programmes"].sum()
        circulated = group["nombre_de_trains_ayant_circule"].sum()
        cancelled = group["nombre_de_trains_annules"].sum()
        delayed = group["nombre_de_trains_en_retard_a_l_arrivee"].sum()
        valid_weather = group.dropna(subset=["weather_stress_score"]).copy()
        weather_score = weighted_average(
            valid_weather["weather_stress_score"],
            valid_weather["nombre_de_trains_programmes"],
        )
        rows.append(
            {
                "date": date,
                "performance_pct": 100 * (circulated - delayed) / circulated,
                "cancellation_pct": 100 * cancelled / programmed,
                "observation_count": int(group["region_current"].nunique()),
                "month": int(date.month),
                "year": int(date.year),
                "mode": "TER",
                "metric_label": "Régularité à 5 minutes",
                "weather_stress_score": weather_score,
                "weather_scope": "Régions TER",
                "nombre_de_trains_programmes": programmed,
                "nombre_de_trains_ayant_circule": circulated,
                "nombre_de_trains_annules": cancelled,
                "nombre_de_trains_en_retard_a_l_arrivee": delayed,
            }
        )

    ter_mode = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    ter_mode["performance_gap"] = add_monthly_gap(ter_mode, "performance_pct", [])
    return ter_mode


ter_mode_monthly = build_ter_mode_monthly(merged)

mode_comparison = pd.concat(
    [
        ter_mode_monthly,
        transilien_monthly,
        intercites_monthly,
    ],
    ignore_index=True,
    sort=False,
).sort_values(["date", "mode"]).reset_index(drop=True)

mode_comparison.to_csv(MODE_COMPARISON_PATH, index=False)

validation_rows = []
for path in [
    TER_CLEAN_PATH,
    TER_MONTHLY_PATH,
    WEATHER_MONTHLY_PATH,
    MERGED_PATH,
    TRANSILIEN_MONTHLY_PATH,
    INTERCITES_MONTHLY_PATH,
    REGION_CONTEXT_PATH,
    MODE_COMPARISON_PATH,
]:
    rows = pd.read_csv(path).shape[0]
    validation_rows.append({"fichier": path.name, "existe": path.exists(), "lignes": rows})

validation = pd.DataFrame(validation_rows)
print("Exports finaux créés :", len(validation_rows))
validation


Exports finaux créés : 8


,fichier,existe,lignes
0,sncf_ter_regularite_mensuelle_clean.csv,True,2296
1,ter_current_regions_monthly.csv,True,1738
2,weather_current_regions_monthly.csv,True,1896
3,ter_weather_current_regions_monthly.csv,True,1738
4,transilien_monthly.csv,True,153
5,intercites_monthly.csv,True,139
6,rail_region_context.csv,True,12
7,rail_mode_comparison_monthly.csv,True,450


## Limites déjà identifiées à ce stade

- `TER`, `Transilien` et `Intercités` n'utilisent pas exactement le même indicateur ;
- la météo est représentée à l'échelle régionale via une ville représentative ;
- l'analyse reste mensuelle, donc on ne peut pas établir une causalité train par train ;
- `Open-Meteo` fournit un historique pratique et reproductible, mais ce n'est pas l'API institutionnelle Météo-France.

Le notebook 2 reprend ensuite ces tables pour l'analyse, les figures et la conclusion finale.
